### A3C（Asynchronous Advantage Actor-Critic，异步优势演员-评论家算法）

**A2C（Advantage Actor-Critic，优势演员-评论家算法）**。
A2C 是 A3C 的 **同步、高效版本**

| 特性 | A3C（Asynchronous，异步） | A2C（Advantage，同步） |
| :--- | :--- | :--- |
| **更新方式** | **异步更新**：每个 Worker 独立计算梯度并更新全局网络。 | **同步更新**：所有 Worker 等待彼此完成，计算平均梯度后统一更新。 |
| **效率** | 理论上更高效，无等待。 | 实践中，由于 GPU 的并行计算优势，A2C 的同步更新通常**效率更高、更稳定**。 |
| **确定性** | 更新具有随机性，可能不稳定。 | 更新是确定性的，训练过程更稳定。 |

### 核心要点
- **A3C（异步）**：多个 Worker 各自为政，独立更新全局网络，理论上更快，但存在竞争和不稳定性。
- **A2C（同步）**：所有 Worker 步调一致，计算平均梯度后再统一更新，充分利用 GPU 算力，实际训练更稳定高效。
A3C算法与A2C算法的不同之处在于，它使用多个代理并行运行，并更新共享模型，以加快学习速度和提高性能。每个Worker直接从Global Network中拿参数，自己与环境互动输出行为。利用每个Worker的梯度，对Global Network的参数进行更新。每一个Worker都是一个A2C。因此，A3C算法可以更快地学习到更好的策略，但也需要更多的计算资源。

**Actor-Critic架构**
- **演员网络 Actor**：负责决策，学习一个策略π(a∣s)，即在状态s下选择动作a的概率
- **Critic（评论家网络）**：负责评估。它根据当前状态估计一个价值（Value），预测从状态s出发能获得的期望回报。它为Actor的决策提供反馈

**3个核心特性**
- **Asynchronous（异步）**：多个智能体独立、并行地与环境交互和更新
- **Advantage（优势）**：使用优势函数（Advantage Function）A(s,a) 来代替原始的奖励。它衡量的是“在状态s下采取动作a，相比该状态的平均水平好多少”，能有效减少策略梯度估计的方差，让学习更稳定。
- **Actor-Critic（演员-评论家）**：结合了基于策略（Policy-based）和基于价值（Value-based）两类方法的优点。

**A3C 组合损失函数**

A3C 通过最小化以下加权组合损失来同时更新 Actor 和 Critic 网络：

$$ 
L_{\text{A3C}} = \underbrace{-\mathbb{E}_t \left[ \log \pi(a_t | s_t) \hat{A}(s_t, a_t) \right]}_{\text{策略损失 (Policy Loss)}} + \underbrace{c_v \cdot \mathbb{E}_t \left[ \left( R_t - V(s_t) \right)^2 \right]}_{\text{价值损失 (Value Loss)}} - \underbrace{c_e \cdot \mathbb{E}_t \left[ H(\pi(\cdot|s_t)) \right]}_{\text{熵正则化 (Entropy Bonus)}}
$$

- **$\pi(a_t | s_t)$**：在当前策略下，状态 $s_t$ 中选择动作 $a_t$ 的概率。
- **$\hat{A}(s_t, a_t)$**：优势函数的估计值。在 A3C 中通常使用 **n-step 回报** 计算：
  $$ \hat{A} = \sum_{i=0}^{n-1} \gamma^i r_{t+i} + \gamma^n V(s_{t+n}) - V(s_t) $$
- **$R_t$**：Critic 网络学习的**目标值**，同样采用 n-step 回报：
  $$ R_t = \sum_{i=0}^{n-1} \gamma^i r_{t+i} + \gamma^n V(s_{t+n}) $$
- **$V(s_t)$**：Critic（评论家）网络对当前状态价值的估计。
- **$H(\pi(\cdot|s_t))$**：策略的熵，计算公式为 $-\sum_a \pi(a|s_t) \log \pi(a|s_t)$。
- **$c_v$ 与 $c_e$**：价值损失和熵正则化的权重系数，用于平衡各部分对总损失的影响。


| 对比维度 | **DQN（Deep Q-Network）** | **A3C（Asynchronous Advantage Actor-Critic）** | **PPO（Proximal Policy Optimization）** |
| :--- | :--- | :--- | :--- |
| **算法类型** | **基于价值（Value-Based）** | **基于策略-价值混合（Actor-Critic）** | **基于策略-价值混合（Actor-Critic）** |
| **核心思想** | 用神经网络近似 Q 表，通过经验回放和目标网络实现稳定训练。 | 多线程异步并行探索，利用 Actor-Critic 和熵正则化提升探索效率。 | 通过**裁剪目标函数**限制策略更新步长，在稳定与效率间取得平衡。 |
| **更新方式** | **离线策略（Off-Policy）**<br>使用经验池中的历史数据更新。 | **在线策略（On-Policy）**<br>使用当前策略与环境交互的数据即时更新。 | **在线策略（On-Policy）**<br>使用当前策略采集的数据进行多轮小批量更新。 |
| **损失函数组成** | **单一批判**：<br>仅优化价值网络的 TD 误差（MSE）。 | **三元复合**：<br>策略损失 + 价值损失 + 熵正则化。 | **裁剪替代目标**：<br>通过 `min(r·A, clip(r)·A)` 限制更新幅度，无需额外正则项。 |
| **探索机制** | **ε-贪心（ε-greedy）**<br>外部强制随机探索，简单但噪声大。 | **熵正则化（Entropy Bonus）**<br>通过增加策略熵激励内部随机性，探索更平滑。 | **隐式约束**<br>通过概率比裁剪限制策略变化幅度，间接控制探索。 |
| **训练架构** | **单智能体 + 经验回放**<br>数据强相关，需经验池打破依赖。 | **异步并行（多线程）**<br>多个智能体独立探索，天然打破数据相关性，无需经验池。 | **单智能体 + 小批量更新**<br>同策略采样，但允许多轮更新提升样本利用率。 |
| **样本效率** | **较高**（经验池可重复利用旧数据）。 | **较低**（数据用完即弃，需不断采集新数据）。 | **中等**（同策略采样，但多轮更新提升利用率）。 |
| **训练稳定性** | **中等**（依赖目标网络冻结稳定目标）。 | **较低**（异步更新可能引入竞争和不稳定性）。 | **极高**（裁剪机制确保更新步长安全可控）。 |
| **超参数敏感度** | **较高**（学习率、目标网络更新频率敏感）。 | **高**（多个线程的学习率、熵系数需精细调参）。 | **较低**（对超参数相对鲁棒，尤其是 clip 范围）。 |
| **硬件需求** | 需要 **GPU** 加速（卷积网络处理图像）。 | **CPU 多线程** 即可，无需 GPU（异步并行天然适合 CPU）。 | **GPU 加速**（小批量数据并行处理效率高）。 |
| **适用场景** | 离散动作空间、图像输入（如 Atari 游戏）。 | 离散/连续动作空间、需要快速探索和并行训练（如机器人控制）。 | 离散/连续动作空间、要求高稳定性和鲁棒性（如 LLM 微调、复杂控制）。 |